In [10]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
load_dotenv()
import os
import asyncio
api_key = os.getenv('OPENAI_API_KEY')
openai_model_client = OpenAIChatCompletionClient(
    model = "gpt-4o",
    api_key=api_key
)

In [11]:
from autogen_agentchat.agents import AssistantAgent

dsa_solver = AssistantAgent(
    name = 'Complex_DSA_Solver',
    model_client=openai_model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words"
)

code_reviewer = AssistantAgent(
    name = 'CODE_REVEIWER',
    model_client=openai_model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized.Give under 10 words."
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=openai_model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required.Give under 10 words. If the code is fine, please say 'TERMINATE'"
)

In [12]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination, TimeoutTermination,TokenUsageTermination

termination_condition = TextMentionTermination('TERMINATE') | MaxMessageTermination(2)

teams = RoundRobinGroupChat(participants=[dsa_solver,code_reviewer,code_editor],termination_condition=termination_condition)

async def test_team():
    task = TextMessage(content = "Write a code to print hello world.",source='User')

    result = await teams.run(task= task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")
        print("\n\n")


await test_team()



User : Write a code to print hello world.



Complex_DSA_Solver : ```python
print("Hello, World!")
```





# Managing State

In [13]:
agent_state  = await dsa_solver.save_state()

In [14]:
agent_state

{'type': 'AssistantAgentState',
 'version': '1.0.0',
 'llm_context': {'messages': [{'content': 'Write a code to print hello world.',
    'source': 'User',
    'type': 'UserMessage'},
   {'content': '```python\nprint("Hello, World!")\n```',
    'thought': None,
    'source': 'Complex_DSA_Solver',
    'type': 'AssistantMessage'}]}}

In [15]:
new_dsa_solver = AssistantAgent(name= "Java_DSA_Solver",model_client=openai_model_client)
new_dsa_solver.load_state(agent_state)

<coroutine object AssistantAgent.load_state at 0x00000108EABCA740>

In [16]:
team_state= await teams.save_state()

In [17]:
team_state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Complex_DSA_Solver': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Write a code to print hello world.',
       'source': 'User',
       'type': 'UserMessage'},
      {'content': '```python\nprint("Hello, World!")\n```',
       'thought': None,
       'source': 'Complex_DSA_Solver',
       'type': 'AssistantMessage'}]}},
   'message_buffer': []},
  'CODE_REVEIWER': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': []}},
   'message_buffer': [{'source': 'User',
     'models_usage': None,
     'metadata': {},
     'created_at': datetime.datetime(2025, 6, 18, 19, 54, 56, 651127, tzinfo=datetime.timezone.utc),
     'content': 'Write a code to print hello world.',
     'type': 'TextMessa